# **Seminar 2. Descriptive Statistics - Solutions to Part 1**

### Case: a trade and investment mission to Belgium

Dataset: `Labour.csv`, 569 Belgian production units, cross-section for 1996.
Source of the data: https://rdrr.io/cran/Ecdat/man/Labour.html
Original reference: Verbeek M. (2004) A Guide to Modern Econometrics, John Wiley and Sons, chapter 4.

| Column | Meaning | Unit |
| --- | --- | --- |
| `capital` | total fixed assets at the end of 1995 | millions of euro |
| `labour` | number of workers | persons |
| `output` | value added | millions of euro |
| `wage` | wage costs per worker | thousands of euro |

How to read this notebook. Every task has four parts:

1. **Task** - the text of the task.
2. **Steps** - what we do, in order.
3. **Code** - the code with a comment on every step. Run the cells from top to bottom with Shift + Enter.
4. **Correct answer** and **Commentary** - the answer to the question and how we got it.

## **0. Setup**

**Steps**

1. Import pandas and numpy.
2. Tell pandas to print numbers with two decimals instead of scientific notation.
3. Load `Labour.csv` into a dataframe called `df`. The file must be in the same folder as the notebook.

In [1]:
# 0. Setup

# Step 1. Import the libraries
import pandas as pd
import numpy as np

# Step 2. Print numbers with two decimals: 11581196.57 instead of 1.158120e+07
pd.set_option("display.float_format", "{:.2f}".format)

# Step 3. Load the data
df = pd.read_csv("Labour.csv")

## **Part A. Preparing the data**

### Task 1. Open the file and check it

**Task.** Load `Labour.csv` into a dataframe. Report the number of rows and columns, the data type
of every column, and the number of missing values in each column. Look at the first ten rows.

*Question:* how many firms are in the sample, and does any column need cleaning before the
analysis starts?

**Steps**

1. `df.shape` gives the number of rows and columns.
2. `df.dtypes` gives the data type of every column.
3. `df.isnull().sum()` counts the missing values in every column.
4. `df.head(10)` shows the first ten rows.

In [2]:
# 1. Check the file

# Step 1. Rows and columns
print("Rows and columns:", df.shape)

# Step 2. Data types
print("\nData types:")
print(df.dtypes)

# Step 3. Missing values in every column
print("\nMissing values:")
print(df.isnull().sum())

# Step 4. The first ten rows
df.head(10)

Rows and columns: (569, 4)

Data types:
capital    float64
labour       int64
output     float64
wage       float64
dtype: object

Missing values:
capital    0
labour     0
output     0
wage       0
dtype: int64


,capital,labour,output,wage
0,2.61,184,9.25,43.08
1,1.32,91,3.66,27.78
2,22.09,426,28.78,44.47
3,10.74,72,4.12,39.73
4,1.16,46,2.89,34.65
5,276.80,7642,480.14,48.94
6,1.68,48,2.15,32.46
7,1.16,108,4.14,31.12
8,0.88,35,2.46,35.12
9,3.49,72,3.85,32.53


**Correct answer: the sample has 569 firms (569 rows) and 4 columns. There are no missing values. No column needs cleaning.**

### Commentary

1. `df.shape` returns `(569, 4)`. One row is one firm, so there are 569 firms.
2. All four columns are numbers. `labour` has the type `int64` (whole numbers), because the number
   of workers is always a whole number. The other three columns have the type `float64` (numbers with decimals).
3. `df.isnull().sum()` returns 0 for every column. This means that no cell is empty.
4. The first ten rows contain only numbers, with no text and no strange symbols.

Conclusion: the data can be used as they are. All statistics below are computed on all 569 firms.

### Task 2. Print the minimum and the maximum of all four columns

*Question:* do any of these eight numbers look like a data entry error rather than a real firm?
State which value surprised you and why you decided to keep it or to doubt it.

**Steps**

1. `df.min()` gives the smallest value of every column.
2. `df.max()` gives the largest value of every column.
3. The minimum of `capital` is printed as 0.00. We print the firms with very small capital to see the exact value.

In [3]:
# 2. The smallest and the largest values

# Step 1. Minimum of every column
print("Minimum:")
print(df.min())

# Step 2. Maximum of every column
print("\nMaximum:")
print(df.max())

# Step 3. Firms with capital below 0.01 million euro (below 10 000 euro).
# We change the display format for this one output, so that small numbers are not shown as 0.00.
pd.set_option("display.float_format", "{:.4f}".format)
print("\nFirms with capital below 0.01 million euro:")
print(df[df["capital"] < 0.01])
pd.set_option("display.float_format", "{:.2f}".format)

Minimum:
capital    0.00
labour     1.00
output     0.03
wage      11.73
dtype: float64

Maximum:
capital    1786.90
labour    10661.00
output     1279.37
wage        188.43
dtype: float64

Firms with capital below 0.01 million euro:
     capital  labour  output    wage
301   0.0022       2  0.3862 42.3892
429   0.0032     118  3.3832 24.9907


**Correct answer: none of the eight numbers is impossible, so all firms stay in the data. The value that surprises most is the minimum of `capital`: about 0.0022 million euro, that is about 2 200 euro of fixed assets.**

### Commentary

1. The minimum of `capital` is printed as 0.00 only because we print two decimals.
   The exact value is 0.0022 million euro, about 2 200 euro.
2. Step 3 shows this firm: 2 workers and a value added of 0.39 million euro. A workshop with two
   workers and very little equipment is possible, so we keep it.
3. Step 3 also shows a second firm: capital of about 3 200 euro, but 118 workers. This combination
   is less likely. It may be an error, or the firm may rent its equipment instead of owning it.
   The data do not allow us to decide, so we keep the firm and note our doubt.
4. The other values are possible:
   - `labour` runs from 1 to 10 661 workers: from a one-person workshop to a large plant;
   - `output` runs from 0.03 to 1 279.37 million euro;
   - `wage` runs from 11.73 to 188.43 thousand euro per worker.
5. An impossible value would be, for example, a negative wage or 0 workers. There are no such values.

Rule: an unusual value is not an error. We delete a value only when it is impossible.

## **Part B. What a typical firm looks like**

### Task 3. Two answers to the same question

**Task.** Compute the mean and the median of all four columns, and then the difference between them.

*Question:* the head of the delegation asks for the size of a typical firm in one number. Do you
report the mean number of workers or the median? Justify the choice in one sentence.

**Steps**

1. `df.mean()` gives the mean (the average) of every column.
2. `df.median()` gives the median of every column: half of the firms have less, half have more.
3. We subtract the median from the mean.

In [4]:
# 3. Mean, median and the difference between them

# Step 1. Mean
print("Mean:")
print(df.mean())

# Step 2. Median
print("\nMedian:")
print(df.median())

# Step 3. Mean minus median
print("\nMean minus median:")
print(df.mean() - df.median())

Mean:
capital    11.52
labour    201.08
output     14.72
wage       38.63
dtype: float64

Median:
capital    2.20
labour    86.00
output     4.60
wage      35.98
dtype: float64

Mean minus median:
capital     9.32
labour    115.08
output     10.12
wage        2.65
dtype: float64


**Correct answer: we report the median, 86 workers, because the mean (201 workers) is raised by a few very large firms and is larger than the size of most firms.**

### Commentary

1. Mean number of workers: 201.08. Median number of workers: 86.
2. The median means: half of the firms have 86 workers or fewer, half have 86 or more.
3. The mean is more than twice the median. The reason: the mean adds up all values, and a few firms
   with thousands of workers (the largest has 10 661) make the sum very large.
4. For all four columns the mean is larger than the median:
   - `capital`: 11.52 against 2.20;
   - `labour`: 201.08 against 86.00;
   - `output`: 14.72 against 4.60;
   - `wage`: 38.63 against 35.98.
5. When the mean is much larger than the median, a few very large values are present.
   In this case the median describes a typical firm better.

A sentence for the delegation: *a typical Belgian manufacturing firm in the sample has about 86 workers,
so most meetings will be with medium-sized firms.*

### Task 4. How wide is the spread

**Task.** For the `labour` column compute the standard deviation, the first and third quartiles and
the interquartile range. Then compute the quartiles of `output`.

*Question:* compare the standard deviation of `labour` with its mean. What does the result tell you
about the sector before you have drawn a single chart?

**Steps**

1. `.std()` gives the standard deviation: how far the values are from the mean, on average.
2. `.quantile(0.25)` gives the first quartile Q1: a quarter of the firms have less.
3. `.quantile(0.75)` gives the third quartile Q3: three quarters of the firms have less.
4. IQR = Q3 - Q1: the width of the middle half of the firms.
5. `.quantile([0.25, 0.50, 0.75])` gives all three quartiles of `output` at once.

In [5]:
# 4. Spread of labour and the quartiles of output

# Step 1. Standard deviation and mean of labour
labour_std = df["labour"].std()
labour_mean = df["labour"].mean()
print("Standard deviation of labour:", round(labour_std, 2))
print("Mean of labour:", round(labour_mean, 2))
print("Standard deviation divided by the mean:", round(labour_std / labour_mean, 2))

# Steps 2 and 3. First and third quartiles of labour
q1 = df["labour"].quantile(0.25)
q3 = df["labour"].quantile(0.75)
print("\nQ1 of labour:", q1)
print("Q3 of labour:", q3)

# Step 4. Interquartile range
print("IQR of labour:", q3 - q1)

# Step 5. The three quartiles of output
print("\nQuartiles of output:")
print(df["output"].quantile([0.25, 0.50, 0.75]))

Standard deviation of labour: 612.0
Mean of labour: 201.08
Standard deviation divided by the mean: 3.04

Q1 of labour: 45.0
Q3 of labour: 176.0
IQR of labour: 131.0

Quartiles of output:
0.25   2.67
0.50   4.60
0.75   9.74
Name: output, dtype: float64


**Correct answer: the standard deviation of `labour` is 612 workers, three times the mean of 201. The firms differ very strongly in size: most firms are small or medium, and a few are very large.**

### Commentary

1. The standard deviation is 612.00, the mean is 201.08. The standard deviation is about 3 times the mean.
2. The number of workers cannot be negative. A standard deviation three times larger than the mean is
   possible only when a few firms are far above the mean. So the sample has a small number of very large firms.
3. The quartiles show where most firms are: Q1 = 45 and Q3 = 176 workers. The middle half of the firms
   (IQR) has between 45 and 176 workers, a range of 131 workers.
4. The quartiles of `output` show the same picture: Q1 = 2.67, median = 4.60, Q3 = 9.74 million euro.
   Three quarters of the firms produce less than 10 million euro, while the largest firm produces 1 279 million euro.

Conclusion: the sector consists of many small and medium firms and a few very large ones.

### Task 5. The shape of the four distributions

**Task.** Compute the skewness of all four columns and rank them from the most asymmetric to the least.

*Question:* which column is closest to symmetric, and why is that the one variable the delegation
can discuss using an average?

**Steps**

1. `df.skew()` gives the skewness of every column.
2. `.sort_values(ascending=False)` puts the largest value first.

In [6]:
# 5. Skewness, from the largest to the smallest

print(df.skew().sort_values(ascending=False))

capital   19.47
output    15.74
labour    12.54
wage       4.10
dtype: float64


**Correct answer: the ranking is capital (19.47), output (15.74), labour (12.54), wage (4.10). The column closest to symmetric is `wage`. It is the only column where the mean (38.63) and the median (35.98) are close, so its average describes a typical firm reasonably well.**

### Commentary

How to read skewness:

- 0 means a symmetric distribution;
- a positive value means a long tail to the right: many small values and a few very large ones;
- the larger the number, the longer the tail.

1. All four values are positive, so all four columns have a long tail to the right.
2. `capital`, `output` and `labour` have very large skewness (from 12.5 to 19.5). For these columns the
   mean is two to five times the median (Task 3), so the mean does not describe a typical firm.
3. `wage` has the smallest skewness, 4.10. Its mean is only 2.65 thousand euro (about 7 per cent)
   above its median.
4. Important: 4.10 is still a clear right skew. `wage` is the closest to symmetric of the four columns,
   but it is not symmetric. The average wage can be quoted, but the median wage, 35.98 thousand euro,
   remains the safer number.

A sentence for the delegation: *the typical wage cost is about 36 thousand euro per worker per year.*

## **Part C. Concentration and outliers**

### Task 6. Apply the outlier rule

**Task.** For the `output` column compute the interquartile range and the bounds Q1 - 1.5 * IQR and
Q3 + 1.5 * IQR. Count how many firms fall outside those bounds and express the count as a share of
the sample.

*Question:* a colleague proposes to delete these firms so that the averages "look normal".
Give one argument against doing so in this particular case.

**Steps**

1. Compute Q1, Q3 and IQR = Q3 - Q1 for `output`.
2. Compute the lower bound Q1 - 1.5 * IQR and the upper bound Q3 + 1.5 * IQR.
3. Select the firms below the lower bound or above the upper bound. These are the outliers.
4. Count them with `len()` and divide by the number of all firms.
5. For the question: compute what share of the total output these firms produce.

In [7]:
# 6. The outlier rule for output

# Step 1. Quartiles and IQR
q1 = df["output"].quantile(0.25)
q3 = df["output"].quantile(0.75)
iqr = q3 - q1
print("Q1:", round(q1, 2), " Q3:", round(q3, 2), " IQR:", round(iqr, 2))

# Step 2. The two bounds
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print("Lower bound:", round(lower, 2), " Upper bound:", round(upper, 2))

# Step 3. Firms outside the bounds
outliers = df[(df["output"] < lower) | (df["output"] > upper)]

# Step 4. Their number and their share of all firms
print("\nNumber of outliers:", len(outliers))
print("Share of all firms, per cent:", round(len(outliers) / len(df) * 100, 1))

# Step 5. Their share of the total output
print("Share of total output, per cent:", round(outliers["output"].sum() / df["output"].sum() * 100, 1))

Q1: 2.67  Q3: 9.74  IQR: 7.08
Lower bound: -7.95  Upper bound: 20.36

Number of outliers: 67
Share of all firms, per cent: 11.8
Share of total output, per cent: 67.0


**Correct answer: IQR = 7.08 million euro, the bounds are -7.95 and 20.36 million euro. 67 firms are outside the bounds, that is 11.8 per cent of the sample. All 67 are above the upper bound. These firms must not be deleted: together they produce 67 per cent of the total output of the sample.**

### Commentary

1. Q1 = 2.666 and Q3 = 9.742, so IQR = 9.742 - 2.666 = 7.08 million euro.
   (The printout rounds Q1 and Q3 to 2.67 and 9.74. pandas computes with the unrounded values.)
2. Lower bound: 2.666 - 1.5 × 7.076 = -7.95. Output cannot be negative, so no firm is below this bound.
3. Upper bound: 9.742 + 1.5 × 7.076 = 20.36. 67 firms produce more than 20.36 million euro.
4. 67 of 569 firms is 11.8 per cent.
5. The argument against deleting them: these 67 firms produce 67 per cent of the total output.
   Without them, the data would describe only one third of Belgian manufacturing. The delegation
   would lose exactly the firms that matter most for a trade and investment mission.

Rule: an outlier is a firm that differs from most others. It is not an error. In this dataset the
outliers are the largest producers of the sector.

### Task 7. Measure the concentration directly

**Task.** Take the 10 per cent of firms with the largest output and compute what share of the total
output of the sample they produce. Use `nlargest` and `sum`.

*Question:* answer question 2 of the delegation in one sentence, quoting your number.

**Steps**

1. Compute 10 per cent of the number of firms: 569 × 0.1 = 56.9. `nlargest` needs a whole number,
   so `int()` drops the decimal part: 56 firms.
2. `df.nlargest(56, "output")` selects the 56 firms with the largest output.
3. Add up their output with `.sum()`.
4. Divide by the total output of all firms and multiply by 100.

In [8]:
# 7. Share of total output produced by the largest 10 per cent of firms

# Step 1. Ten per cent of the firms, as a whole number
n_top = int(len(df) * 0.1)
print("Number of firms in the top 10 per cent:", n_top)

# Step 2. The firms with the largest output
top_firms = df.nlargest(n_top, "output")

# Step 3. Their total output
top_output = top_firms["output"].sum()
total_output = df["output"].sum()
print("Output of the top firms, million euro:", round(top_output, 1))
print("Output of all firms, million euro:", round(total_output, 1))

# Step 4. Their share of the total output
print("Share of total output, per cent:", round(top_output / total_output * 100, 1))

Number of firms in the top 10 per cent: 56
Output of the top firms, million euro: 5368.6
Output of all firms, million euro: 8375.2
Share of total output, per cent: 64.1


**Correct answer: the 56 largest firms (10 per cent of the sample) produce 64.1 per cent of the total output, about two thirds.**

### Commentary

1. The 56 largest firms produce 5 368.6 million euro out of 8 375.2 million euro.
2. 5 368.6 / 8 375.2 × 100 = 64.1 per cent.
3. The other 513 firms (90 per cent) produce together only about 36 per cent.
4. If we round 56.9 up to 57 firms, the share is 64.4 per cent. The conclusion does not change.

A sentence for the delegation (question 2): *the sector is highly concentrated: 10 per cent of the firms
produce about two thirds of the output, so the mission programme should be built around these large producers.*

Compare Tasks 3 and 7. A typical firm has 86 workers (Task 3), but most of the output comes from a few
large firms (Task 7). The delegation will meet mostly medium-sized firms, while the main economic
weight is in a short list of large ones.

## **Summary: answers to the delegation**

1. **A typical firm** has about 86 workers (median). The mean of 201 workers is raised by a few very large firms.
2. **Concentration**: 10 per cent of the firms produce 64 per cent of the output. 67 firms are outliers by the 1.5 × IQR rule, and they produce 67 per cent of the output.
3. **Wage level**: the median wage cost is about 36 thousand euro per worker per year (mean 38.6).
4. **Wages and productivity**: Part 1 does not answer this question. It needs the correlation, which is not part of these tasks.

Methods used:

- check the data: `shape`, `dtypes`, `isnull().sum()`, `head()`, `min()`, `max()`;
- centre: `mean()` and `median()`; when the mean is much larger than the median, use the median;
- spread: `std()`, `quantile()`, IQR;
- shape: `skew()`;
- outliers: the 1.5 × IQR rule;
- concentration: `nlargest()` and `sum()`.